In [1]:
import json
import os
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from numpy.typing import NDArray

from phi_3_5_constants import hidden_state_size, seed, dsets_folder, token_lengths_path, train_split_records_path, \
    validation_split_records_path
from phi_3_5_probe import learn_directions_and_train_probe, PolarityAwareTruthProbe

In [ ]:
np_rng = np.random.default_rng(seed)
torch_rng = torch.Generator().manual_seed(seed)
torch.manual_seed(seed)

In [5]:
activations_data_folder= Path("D:\\TruthIsUniversal_In_Phi_3_5_Mini\\best2_layers_activations_for_final_token_of_sequences")
results_folder = Path("summary_vectors")

In [ ]:
results_folder.mkdir(exist_ok=True)

In [6]:
dsets_index_df = pd.read_csv(dsets_folder / "datasets_index.csv", index_col="Idx")

dict_keys(['token_lengths', 'post6_hidden_states', 'post16_hidden_states', 'post22_hidden_states', 'post29_hidden_states'])

In [ ]:
num_dsets = dsets_index_df.shape[0]

In [7]:
with token_lengths_path.open("r") as f:
    record_lengths_in_tokens = json.load(f)
with train_split_records_path.open("r") as f:
    train_split_record_idxs = json.load(f)
with validation_split_records_path.open("r") as f:
    validation_split_record_idxs = json.load(f)

torch.Size([500, 31, 3072])

In [ ]:
all_dsets_activations: list[torch.Tensor] = []
all_dsets_labels: list[NDArray] = []

In [2]:
train_data_lyr18_avg_activs_by_dset = torch.zeros((num_dsets, hidden_state_size))
train_data_lyr25_avg_activs_by_dset = torch.zeros((num_dsets, hidden_state_size))

#0th dimension is indexed by dataset id; "train_data" means only based on the train segment of the dataset
train_data_lyr18_truth_dirs: torch.Tensor = torch.zeros((num_dsets, hidden_state_size))
train_data_lyr18_polarity_dirs: torch.Tensor = torch.zeros((num_dsets, hidden_state_size))
# TODO do the same for layer25 activations (and later also layer18+layer25 concatenated activations) after confirming that the learned truth/polarity directions for layer 18 on a given dataset can effectively be used by a linear classifier to predict truth/falsehood in validation segment of same dataset
train_data_lyr25_truth_dirs: torch.Tensor = torch.zeros((num_dsets, hidden_state_size))
train_data_lyr25_polarity_dirs: torch.Tensor = torch.zeros((num_dsets, hidden_state_size))

lyr18_probes: list[PolarityAwareTruthProbe] = []
lyr25_probes: list[PolarityAwareTruthProbe] = []

In [4]:
for dset_idx, dset_dtls in dsets_index_df.iterrows():
    categ_nm = dset_dtls["Categ_Folder"]
    dset_file_nm = dset_dtls["Dataset_File"]
    dset_nm = os.path.splitext(dset_file_nm)[0]
    
    dataset = pd.read_csv(dsets_folder / categ_nm / dset_file_nm)
    dset_size = dataset.shape[0]
    all_dsets_labels.append(dataset['label'].to_numpy())
    
    activs_path = activations_data_folder / categ_nm / (dset_nm + ".pt")
    relevant_activations = torch.load(activs_path, weights_only=True)
    assert relevant_activations.shape == (2, dset_size, hidden_state_size)
    all_dsets_activations.append(relevant_activations)
    
    train_split_activs = relevant_activations[:, train_split_record_idxs[str(dset_idx)], :]
    train_split_truth_labels = dataset[train_split_record_idxs[str(dset_idx)]]["label"].to_numpy()
    
    train_split_polarity_labels = np.ones(train_split_truth_labels.shape)
    if dset_dtls["is_negated"]:
        train_split_polarity_labels *= -1
    
    val_split_activs = relevant_activations[:, validation_split_record_idxs[str(dset_idx)], :]
    val_split_truth_labels = dataset[validation_split_record_idxs[str(dset_idx)]]["label"].to_numpy()
    
    print(f"doing direction-learning and probe training for layer 18 of dataset {dset_nm} in category {categ_nm}")
    lyr18_probe = learn_directions_and_train_probe(
        train_split_activs[0,:,:], train_split_truth_labels, train_split_polarity_labels, val_split_activs[0,:,:], val_split_truth_labels, np_rng)
    
    train_data_lyr18_truth_dirs[dset_idx, :] = lyr18_probe.truth_dir.clone().cpu().T
    train_data_lyr18_polarity_dirs[dset_idx, :] = lyr18_probe.polarity_dir.clone().cpu().T
    train_data_lyr18_avg_activs_by_dset[dset_idx, :] = lyr18_probe.mean_activation.clone().cpu().T
    lyr18_probes.append(lyr18_probe)
    
    print(f"doing direction-learning and probe training for layer 25 of dataset {dset_nm} in category {categ_nm}")
    lyr25_probe = learn_directions_and_train_probe(
        train_split_activs[1,:,:], train_split_truth_labels, train_split_polarity_labels, val_split_activs[1,:,:], val_split_truth_labels, np_rng)
    
    train_data_lyr25_truth_dirs[dset_idx, :] = lyr25_probe.truth_dir.clone().cpu().T
    train_data_lyr25_polarity_dirs[dset_idx, :] = lyr25_probe.polarity_dir.clone().cpu().T
    train_data_lyr25_avg_activs_by_dset[dset_idx, :] = lyr25_probe.mean_activation.detach().cpu().T
    lyr25_probes.append(lyr25_probe)

In [10]:
#TODO train on positive + negated dataset pairs (within a category) to confirm whether polarity direction is doing any good 

np.int64(13)

tensor([-0.2372,  0.4232, -0.1990,  ..., -0.5974,  0.0763, -0.2289])

tensor([[ 1.6488e-01, -1.2801e-01,  4.0603e-01,  ...,  6.4095e-02,
          1.9822e-01, -2.8558e-01],
        [-3.0338e-02,  8.3794e-02,  1.1366e-01,  ..., -1.0596e-01,
          5.3798e-02,  1.1725e-02],
        [ 1.6687e-01,  8.1258e-03, -3.4267e-02,  ..., -9.6795e-02,
          4.5040e-02,  1.0863e-01],
        ...,
        [-2.3185e-02,  2.7375e-02, -1.9377e-04,  ...,  8.0140e-02,
          1.2388e-01, -1.3121e-01],
        [ 1.2641e+00,  2.6569e-01, -6.2784e-01,  ..., -3.1791e-01,
          6.3754e-01,  2.0045e+00],
        [-1.5921e-01,  2.9449e-01, -2.5346e-01,  ..., -6.1642e-01,
          4.9549e-02, -2.0675e-01]])

In [ ]:
# TODO confirm that the learned truth/polarity directions for layer 18 on a given dataset can effectively be used by a linear classifier to predict truth/falsehood in other datasets in group